# лексический канал: BM25 с лемматизацией

2ой независимый источник кандидатов. считает совпадение слов а не смысла поэтому должен ловить то, что семантический энкодер размазывает: точные термины, редкие слова, числа в названиях. 

валидацию и корпус берем готовые из 2ого блокнота

In [1]:
import sys, time
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import polars as pl

from src.params import parse_item
from src.text import normalize
from src.metrics import recall_at_k, recall_by_slice
from src.lexical import tokenize, Bm25Fields

PROC = Path("../data/processed")
val = pl.read_parquet(PROC / "val_queries.parquet")
corpus = pl.read_parquet(PROC / "val_corpus.parquet")
print("запросов:", len(val), " объявлений:", len(corpus))

/Users/amirhamidullin/PycharmProjects/avito_recsys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


запросов: 2500  объявлений: 191852


## 3 поля вместо одного текста

заголовок, список услуг из прайс листа и описание индексируем отдельно, чтобы дать им разные веса. слово в заголовке значит больше чем то же слово в середине описания

In [2]:
parsed = [parse_item(p) for p in corpus["item_infm_params_text"]]

fields = {"title": corpus["item_title_raw"].to_list(), "services": [", ".join(p["services"]) + " " + p["vid"] + " " + p["tip"] for p in parsed], "description": corpus["item_description_raw"].fill_null("").to_list()}
print("у скольких объявлений есть список услуг", sum(1 for p in parsed if p["services"]), "из", len(parsed))
print("title", fields["title"][0])
print("services", fields["services"][0][:200])
print("description", fields["description"][0][:200])

у скольких объявлений есть список услуг 111438 из 191852
title Сварка, монтаж оптоволокна волс
services  Компьютерная помощь Интернет и другие сети
description Срочный ремонт оптоволоконного кабеля.
Сварка и монтаж кабеля ВОЛС. 
Тестирование оптических волокон рефлектометром. 
Оптические компоненты для ремонта и монтажа всегда в наличии. 
Поиск и устранение 


## токезнизация и индекс

лемматизация через pymorphy с кэшем по словам: уникальных слов на порядки меньше, чем всех так что 190к описаний проходят в теории за минкты

In [3]:
t = time.time()
print("пример токенов запроса", tokenize("Подъемник 4 х стоечный для автосервиса"))
print("пример токенов заголовка", tokenize(fields["title"][0]))

bm25 = Bm25Fields({"title": 3.0, "services": 2.0, "description": 1.0})
bm25.build(fields)
print("индекс построен за", round((time.time() - t) / 60, 1), "минут")

пример токенов запроса ['подъемник', 'стоечный', 'автосервис']
пример токенов заголовка ['сварка', 'монтаж', 'оптоволокно', 'волс']


индекс построен за 0.8 минут


## считаем recall

In [4]:
queries = [normalize(q) for q in val["search_query"]]
item_ids = corpus["item_id"].to_list()
relevant = {qid: set(rel) for qid, rel in zip(val["query_id"], val["relevant"])}
is_seen = dict(zip(val["query_id"], val["is_seen"]))
has_filter = dict(zip(val["query_id"], val["search_infm_params_text"].str.strip_chars() != ""))

t = time.time()
idx = bm25.topk(queries, k=200)
print("поиск по 2500 запросам занял", round(time.time() - t), "секунд\n")

predicted = {qid: [item_ids[j] for j in row] for qid, row in zip(val["query_id"], idx)}
for k in [10, 50, 100, 200]:
    print("recall@" + str(k), round(recall_at_k(predicted, relevant, k), 4))

print("\nrecall@50 по срезам")
for name, flags in [("текст запроса был в трейне", is_seen), ("в запросе есть фильтр", has_filter)]:
    for value, (r, n) in recall_by_slice(predicted, relevant, flags, 50).items():
        print(name, "=", value, round(r, 4), "запросов", n, "")

поиск по 2500 запросам занял 3 секунд

recall@10 0.1602
recall@50 0.3394
recall@100 0.4364
recall@200 0.5481

recall@50 по срезам
текст запроса был в трейне = False 0.3763 запросов 1575 
текст запроса был в трейне = True 0.2767 запросов 925 
в запросе есть фильтр = False 0.3915 запросов 1142 
в запросе есть фильтр = True 0.2957 запросов 1358 


## Промахи

Где лексика не справилась: обычно это синонимы и перефразировки, ровно то, что должен закрывать dense канал

In [5]:
title_by_id = dict(zip(corpus["item_id"], corpus["item_title_raw"]))
misses = [qid for qid in relevant if not (set(predicted[qid][:50]) & relevant[qid])]
print("промахов в топ-50", len(misses), "из", len(relevant), "\n")

rng = np.random.default_rng(0)
for qid in rng.choice(misses, size=6, replace=False):
    row = val.filter(pl.col("query_id") == qid).row(0, named=True)
    print("запрос", row["search_query"])
    print("выбрано", [title_by_id[i] for i in relevant[qid]][:2])
    print("bm25 поставил первыми", [title_by_id[i] for i in predicted[qid][:3]])

промахов в топ-50 1613 из 2500 

запрос перевозка мелкогабаритных грузов
выбрано ['Грузоперевозки по городу/межгород/помощь на дороге']
bm25 поставил первыми ['Перевозки мелкогабаритного груза', 'Доставка мелкогабаритных грузов Севастополь', 'Перевозка груза']
запрос трансфер 10 мест
выбрано ['Аренда микроавтобуса с водителем и без']
bm25 поставил первыми ['Трансфер до 10 мест', 'Трансфер 10-19 мест', 'Трансфер минивен 8 мест']
запрос проект для школу
выбрано ['Курсовые/Дипломы/Диссертации и др']
bm25 поставил первыми ['Исследовательские проекты для школы', 'Индивидуальные проекты для школы', 'Исследовательские проекты для школы']
запрос дипломная работа маркшейдерское дело
выбрано ['Дипломная работа / Диссертация / ВКР']
bm25 поставил первыми ['Маркшейдерские и геодезические работы', 'Курсовые работы, Дипломные работы, ВКР', 'Дипломная работа']
запрос ремонт пылесосов tefal
выбрано ['Ремонт мелкой бытовой техники парогенераторов']
bm25 поставил первыми ['Ремонт пылесосов, ремонт dyson

## Сохраняем кандидатов

Топ-200 по каждому запросу пригодится при объединении каналов и как признак реранкера

In [6]:
cands = PROC / "cands"
cands.mkdir(exist_ok=True)
np.save(cands / "bm25_val_top200.npy", idx)
print("сохранено:", idx.shape)

сохранено: (2500, 200)
